# **Data Exploration**

In [1]:
# Import the required libraries
import pandas as pd
import numpy as np

#Load the dataset
df = pd.read_csv("retail_sales_raw.csv")

# Display the dataset
print(df.head())

   Transaction ID Customer ID Product Category    Product Name  Quantity  \
0          100932       C0503        Groceries  Cooking Oil 1L       2.0   
1          101309       C0339     Home&Kitchen      Frying Pan       3.0   
2          100397       C0535         Clothing        Sneakers       4.0   
3          100538       C0224      Electronics     USB-C Cable       3.0   
4          100791       C0223   Home & Kitchen      Frying Pan       5.0   

   Unit Price  Total Amount      Purchase Date Customer City Payment Method  
0     1729.13       3458.26    August 16, 2024        Quetta     Debit Card  
1    10892.87      32678.61         2025-09-09      Peshawar  Mobile Wallet  
2     2668.91      10675.64  December 10, 2024     hyderabad  Bank Transfer  
3    72429.71     217289.13         2025-12-28     islamabad    Credit Card  
4     6636.68      33183.40     April 15, 2025        Quetta    Credit Card  


In [2]:
# Shape of dataset
print("Shape:", df.shape)

Shape: (1730, 10)


In [3]:
# Column names
print("\nColumns:", df.columns.tolist())


Columns: ['Transaction ID', 'Customer ID', 'Product Category', 'Product Name', 'Quantity', 'Unit Price', 'Total Amount', 'Purchase Date', 'Customer City', 'Payment Method']


In [4]:
# Inspecting the dirtiness in data
print("=== MISSING VALUES ===")
print(df.isnull().sum())

print("\n=== DUPLICATE ROWS ===")
print("Duplicates:", df.duplicated().sum())

print("\n=== CATEGORY VARIANTS (should be 6 clean ones) ===")
print(df["Product Category"].value_counts())

print("\n=== PAYMENT METHOD VARIANTS ===")
print(df["Payment Method"].value_counts(dropna=False))

print("\n=== CITY VARIANTS ===")
print(df["Customer City"].value_counts(dropna=False))

print("\n=== DATA TYPES ===")
print(df.dtypes)

=== MISSING VALUES ===
Transaction ID       0
Customer ID          0
Product Category     0
Product Name         0
Quantity            34
Unit Price          24
Total Amount         0
Purchase Date        0
Customer City       69
Payment Method      60
dtype: int64

=== DUPLICATE ROWS ===
Duplicates: 30

=== CATEGORY VARIANTS (should be 6 clean ones) ===
Product Category
Beauty            282
Groceries         273
Sports            270
Electronics       268
Home & Kitchen    267
Clothing          232
home & kitchen     15
BEAUTY             15
 Sports            15
 Electronics       14
Home&Kitchen       12
Grocery            12
sports             10
groceries          10
clothing           10
beauty              8
ELECTRONICS         8
 Clothing           6
electronics         3
Name: count, dtype: int64

=== PAYMENT METHOD VARIANTS ===
Payment Method
Bank Transfer    328
Cash             319
Credit Card      308
Debit Card       307
Mobile Wallet    293
NaN               60
mobile w

# **Data Cleaning**

# Removing Duplicates:

In [5]:
# Remove duplicate rows
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f"Removed {before - after} duplicate rows. {before} → {after}")

Removed 30 duplicate rows. 1730 → 1700


# Standardize Text Columns:

In [6]:
# Standardize text columns to fix inconsistency

# Category: strip whitespace, then map variants to clean names
df["Product Category"] = df["Product Category"].str.strip().str.title()

# Fix the special cases that .title() won't handle correctly
cat_map = {
    "Home&Kitchen": "Home & Kitchen",
    "Home & Kitchen": "Home & Kitchen",
    "Grocery": "Groceries",
    "Groceries": "Groceries",
}
df["Product Category"] = df["Product Category"].replace(cat_map)

print(df["Product Category"].value_counts())

Product Category
Beauty            302
Groceries         289
Sports            289
Home & Kitchen    288
Electronics       288
Clothing          244
Name: count, dtype: int64


In [7]:
# City: strip whitespace and title-case
df["Customer City"] = df["Customer City"].str.strip().str.title()
print(df["Customer City"].value_counts(dropna=False))

Customer City
Faisalabad    180
Multan        174
Quetta        167
Islamabad     167
Rawalpindi    167
Karachi       165
Lahore        165
Sialkot       160
Hyderabad     153
Peshawar      134
NaN            68
Name: count, dtype: int64


In [8]:
# Payment Method: map all variants to clean labels
pay_map = {
    "credit card": "Credit Card", "cc": "Credit Card", "Credit card": "Credit Card",
    "debit card": "Debit Card", "dc": "Debit Card",
    "cash": "Cash", "CASH": "Cash",
    "mobile wallet": "Mobile Wallet", "Wallet": "Mobile Wallet",
    "bank transfer": "Bank Transfer", "Transfer": "Bank Transfer",
}
# strip first, then map; anything already clean stays as-is
df["Payment Method"] = df["Payment Method"].str.strip().replace(pay_map)
print(df["Payment Method"].value_counts(dropna=False))

Payment Method
Bank Transfer    343
Cash             334
Credit Card      318
Debit Card       314
Mobile Wallet    313
NaN               59
DC                14
CC                 5
Name: count, dtype: int64


# Standardize Dates:

In [9]:
df["Purchase Date"] = pd.to_datetime(df["Purchase Date"], format="mixed", dayfirst=True, errors="coerce")

print("Date range:", df["Purchase Date"].min(), "to", df["Purchase Date"].max())
print("Failed to parse (NaT):", df["Purchase Date"].isna().sum())
print(df["Purchase Date"].dtype)

Date range: 2024-01-03 00:00:00 to 2025-12-28 00:00:00
Failed to parse (NaT): 0
datetime64[ns]


# Data Type Fix On Numeric Columns:

In [10]:
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Unit Price"] = pd.to_numeric(df["Unit Price"], errors="coerce")
df["Total Amount"] = pd.to_numeric(df["Total Amount"], errors="coerce")
print(df.dtypes)

Transaction ID               int64
Customer ID                 object
Product Category            object
Product Name                object
Quantity                   float64
Unit Price                 float64
Total Amount               float64
Purchase Date       datetime64[ns]
Customer City               object
Payment Method              object
dtype: object


# Handle Outliers:

In [11]:
# Flag the impossible rows
bad = df[(df["Quantity"] <= 0) | (df["Unit Price"] <= 0)]
print(f"Impossible-value rows found: {len(bad)}")
print(bad[["Quantity","Unit Price"]])

# Decision: set them to NaN so they're handled as missing (defensible and documented)
df.loc[df["Quantity"] <= 0, "Quantity"] = np.nan
df.loc[df["Unit Price"] <= 0, "Unit Price"] = np.nan

Impossible-value rows found: 8
      Quantity  Unit Price
131        5.0        0.00
378       -1.0     4715.10
424       -3.0    41294.24
992       -2.0     1244.62
1289       1.0        0.00
1305       4.0        0.00
1639       5.0        0.00
1720      -1.0     3231.32


# Handle Missing Values:

In [12]:
# Quantity: fill missing with the median (whole number, robust to outliers)
df["Quantity"] = df["Quantity"].fillna(df["Quantity"].median())

# Unit Price: fill missing with the median price *within the same category* (more accurate than overall median)
df["Unit Price"] = df.groupby("Product Category")["Unit Price"].transform(
    lambda s: s.fillna(s.median())
)

# Customer City: categorical, no sensible average — fill with "Unknown"
df["Customer City"] = df["Customer City"].fillna("Unknown")

# Payment Method: same logic — fill with "Unknown"
df["Payment Method"] = df["Payment Method"].fillna("Unknown")

print(df.isnull().sum())

Transaction ID      0
Customer ID         0
Product Category    0
Product Name        0
Quantity            0
Unit Price          0
Total Amount        0
Purchase Date       0
Customer City       0
Payment Method      0
dtype: int64


# Recalculate Total Amount To Fix Broken Relationship:

In [13]:
# Check how many were inconsistent before fixing
df["expected_total"] = (df["Quantity"] * df["Unit Price"]).round(2)
mismatch = (abs(df["Total Amount"] - df["expected_total"]) > 0.01).sum()
print(f"Rows where Total Amount != Quantity x Unit Price: {mismatch}")

# Recalculate Total Amount as the source of truth
df["Total Amount"] = df["expected_total"]
df = df.drop(columns=["expected_total"])
print("Total Amount recalculated for all rows.")

Rows where Total Amount != Quantity x Unit Price: 143
Total Amount recalculated for all rows.


# Fixing Leftovers:

In [14]:
# Fix leftover payment abbreviations
df["Payment Method"] = df["Payment Method"].replace({"CC": "Credit Card", "DC": "Debit Card"})

# Round Unit Price and Total Amount to clean 2-decimal currency
df["Unit Price"] = df["Unit Price"].round(2)
df["Total Amount"] = (df["Quantity"] * df["Unit Price"]).round(2)

# Verify
print(df["Payment Method"].value_counts())
print("Unit Price sample:", df["Unit Price"].head().tolist())

Payment Method
Bank Transfer    343
Cash             334
Debit Card       328
Credit Card      323
Mobile Wallet    313
Unknown           59
Name: count, dtype: int64
Unit Price sample: [1729.13, 10892.87, 2668.91, 72429.71, 6636.68]


# **Final Verification And Export**

In [15]:
print("=== FINAL CHECK ===")
print("Shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Duplicates:", df.duplicated().sum())
print("Categories:", df["Product Category"].nunique(), "→", sorted(df["Product Category"].unique()))
print("Payment methods:", sorted(df["Payment Method"].unique()))
print("Date type:", df["Purchase Date"].dtype)
print("Total = Qty*Price for all rows:", (abs(df["Total Amount"] - df["Quantity"]*df["Unit Price"]) < 0.01).all())

# Export the cleaned dataset
df.to_csv("retail_sales_cleaned.csv", index=False)
print("\nSaved retail_sales_cleaned.csv")

=== FINAL CHECK ===
Shape: (1700, 10)
Missing values: 0
Duplicates: 0
Categories: 6 → ['Beauty', 'Clothing', 'Electronics', 'Groceries', 'Home & Kitchen', 'Sports']
Payment methods: ['Bank Transfer', 'Cash', 'Credit Card', 'Debit Card', 'Mobile Wallet', 'Unknown']
Date type: datetime64[ns]
Total = Qty*Price for all rows: True

Saved retail_sales_cleaned.csv
